In [1]:
from funzioni_varie.function_log import *

In [2]:
model_emb = "text-embedding-3-large"

path_embedding = "IMU Embeddings" + "/" + model_emb
metadata_path = path_embedding + "/metadata.json"

# Carica metadata
with open(metadata_path, "r", encoding="utf-8") as f:
    all_metadata = json.load(f)

with open("IMU embeddings/vocabolario.json", "r", encoding="utf-8") as f:
    vocabulary = json.load(f)

In [3]:
last_logs = get_log()

In [4]:
get_history(logs= last_logs)

In [5]:
last_logs

[{'request_id': '577082af-413e-4315-b7b2-c279af039976',
  'timestamp': '2026-03-12 11:28:50',
  'model': 'gpt-5-mini',
  'user_query': "Devo pagare per l'abitazione principale?",
  'model_steps': [{'iteration': 0,
    'response_id': 'resp_0022f71c8a2f5dc70069b2956256148194a4536c075e37c7ce',
    'previous_response_id': None},
   {'iteration': 1,
    'response_id': 'resp_0022f71c8a2f5dc70069b2956441348194a78680d731141989',
    'previous_response_id': 'resp_0022f71c8a2f5dc70069b2956256148194a4536c075e37c7ce'}],
  'tool_calls': [{'iteration': 1,
    'tool_call_number': 1,
    'tool_name': 'retrieve_normativa',
    'args': {'tipologia': 'Legge',
     'anno_legge': 2019,
     'reference': {'comma': 740},
     'suffix': None,
     'year': 2026},
    'result': [269]}],
  'internal_messages': [],
  'final_answer': "## Regola generale\nL'**abitazione principale** è **esente dall'IMU**.  \n\n---\n\n### Eccezione importante\nSe l'unità abitativa è classificata nelle categorie catastali **A/1, A/8 

In [6]:
text_normativa(all_metadata= all_metadata,emb_ids=[417])

{'documento': 'decreto-legge 31 maggio 1994, n. 330 semplificazione di talune disposizioni in materia tributaria.',
 'paragrafi': ['Art. 6', 'Comma 8'],
 'testo': '8. il pagamento di ritenute alla fonte, di imposte, di tasse e contributi erariali, regionali e locali il cui termine cade di sabato o di giorno festivo è considerato tempestivo se effettuato il primo giorno lavorativo successivo'}

In [7]:
all_titles = list(set([el['metadata']['document'] for el in all_metadata]))

# Regex per estrarre tipo + data + numero
pattern = re.compile(
    r"(legge|decreto(?:-legge)?|decreto legislativo|decreto ministeriale) "  # Tipo atto
    r"(?:del\s*)?"  # opzionale "del"
    r"(\d{1,2} [a-z]+ \d{4}|\d{2}/\d{2}/\d{4})"  # Data in formato "30 dicembre 1992" o "04/05/2023"
    r"(?:, n\. \d+)?"  # Numero legge/decreto opzionale
    , flags=re.IGNORECASE
)

map_documents = {}

for testo in all_titles:
    match = pattern.search(testo)
    if match:
        # Combiniamo tipo + data + numero se presente
        estratto = match.group(0)
        map_documents[testo] = estratto

In [8]:
map_documents

{'decreto-legge 31 maggio 1994, n. 330 semplificazione di talune disposizioni in materia tributaria.': 'decreto-legge 31 maggio 1994, n. 330',
 'regio decreto 16 marzo 1942, n. 262 approvazione del testo del codice civile. (042u0262)': 'decreto 16 marzo 1942, n. 262',
 'decreto ministeriale del 04/05/2023': 'decreto ministeriale del 04/05/2023',
 'legge 20 maggio 1985, n. 222 disposizioni sugli enti e beni ecclesiastici in italia e per il sostentamento del clero cattolico in servizio nelle diocesi.': 'legge 20 maggio 1985, n. 222',
 "decreto legislativo 30 dicembre 1992, n. 504 riordino della finanza degli enti territoriali, a norma dell'articolo 4 della legge 23 ottobre 1992, n. 421.": 'decreto legislativo 30 dicembre 1992, n. 504',
 "legge 27 dicembre 2019, n. 160 bilancio di previsione dello stato per l'anno finanziario 2020 e bilancio pluriennale per il triennio 2020-2022. (19g00165)": 'legge 27 dicembre 2019, n. 160'}

In [9]:
last_logs[0].get("tool_calls")

[{'iteration': 1,
  'tool_call_number': 1,
  'tool_name': 'retrieve_normativa',
  'args': {'tipologia': 'Legge',
   'anno_legge': 2019,
   'reference': {'comma': 740},
   'suffix': None,
   'year': 2026},
  'result': [269]}]

In [10]:
all_chunks = get_fonti(map_documents= map_documents, all_metadata= all_metadata, vocabulary= vocabulary, logs= last_logs, index= 0)

In [11]:
len(all_chunks)

1

In [12]:
pair_voc = show_titles_for_widget(map_documents= map_documents, all_metadata= all_metadata, vocabulary= vocabulary, logs= last_logs, index= 0 #, only_titles= True
                       )

In [13]:
pair_voc

{'retrieve_normativa': {'legge 27 dicembre 2019, n. 160': [(['Art. 1',
     'Comma 740'],
    "740. il presupposto dell'imposta è il possesso di immobili. il possesso dell'abitazione principale o assimilata, come definita alle lettere b) e c) del comma 741, non costituisce presupposto dell'imposta, salvo che si tratti di un'unità abitativa classificata nelle categorie catastali a/1, a/8 o a/9.")]}}

In [14]:
show_retrieve_widget(pair_voc)

Dropdown(description='Retrieve:', layout=Layout(width='80%'), options=(('retrieve_normativa', 'retrieve_normat…